# Assignment 2: Can Experts Forecast NFL Games?
ISA 414, Trey Patrick

## Question 1

In [ ]:
def collect_data(url):
    import requests
    import re
    import pandas as pd
    r = requests.get(url)
    html_code = r.text

    # Each team row: a "game" cell (team name) followed by a "pick" cell (probability)
    hits = re.finditer(r"<td class=\"game\">.+?</td>\s*<td class=\"pick\">.+?</td>", html_code)

    teams = []
    picks = []
    winners = []
    for hit in hits:
        row = hit.group()

        # Team name sits between the link and </a>; the winner's name is wrapped in <b></b>
        team = re.search(r"html\">(.+?)</a>", row).group(1)
        if re.search(r"<b>", team) is not None:
            winners.append(True)
        else:
            winners.append(False)
        team = re.sub(r"<b>", "", team)
        team = re.sub(r"</b>", "", team)
        teams.append(team)

        # Probability: the digits right before the % sign (works whether or not the pick is bold)
        pick = re.search(r"(\d+)%", row).group(1)
        picks.append(int(pick))

    # Home team is always listed first, so the rows alternate: home, away, home, away, ...
    rows = []
    for index in range(0, len(teams), 2):
        home_team = teams[index]
        away_team = teams[index + 1]
        if winners[index]:
            winner_team = home_team
        else:
            winner_team = away_team
        rows.append([home_team, away_team, winner_team, picks[index], picks[index + 1]])

    result = pd.DataFrame(rows, columns = ["Home_Team", "Away_Team", "Winner_Team",
                                           "Prediction_Home_Team", "Prediction_Away_Team"])
    return result

In [ ]:
collect_data("https://rayna-xu-is.github.io/probabilityfootball/page1.html")

## Question 2

In [ ]:
import pandas as pd

weekly_data = []
for week in range(1, 18):
    weekly_data.append(collect_data(f"https://rayna-xu-is.github.io/probabilityfootball/page{week}.html"))

season_data = pd.concat(weekly_data, ignore_index = True)
season_data.shape

## Question 3

In [ ]:
from scipy import stats

predictions_winner = []
for index in range(len(season_data)):
    if season_data["Winner_Team"][index] == season_data["Home_Team"][index]:
        predictions_winner.append(season_data["Prediction_Home_Team"][index])
    else:
        predictions_winner.append(season_data["Prediction_Away_Team"][index])

print("Number of games:", len(predictions_winner))
print("Mean prediction for the winning team:", sum(predictions_winner) / len(predictions_winner))
stats.ttest_1samp(predictions_winner, 50, alternative = "greater")

**Which forecaster is more accurate?** On average, the experts assigned about 56.8% probability to the team that actually won, compared with 50% for an unbiased coin. The one-sided t-test gives t ≈ 6.21 and a p-value of about 1.06 × 10⁻⁹, far below 0.05, so we reject the null hypothesis that the mean is 50. The experts are more accurate on average than a coin.

**Should the group use expert-driven forecasts instead of model-based forecasts?** Not on this evidence alone. The result shows experts are better than random, but beating a coin is a low bar. It does not show that experts beat a statistical model built on historical data. Before switching, the experts should be compared against a model-based forecast on the same games. Expert forecasts could still be valuable where historical data are scarce, such as preseason predictions, or combined with a model.

**Is an unbiased coin an appropriate baseline?** It is a useful minimum check: a forecaster that cannot beat a coin has no skill. But it is too weak to be the main benchmark. Simple strategies already beat 50%, like always picking the home team or the team with the better record last season. A fairer baseline would be one of those simple rules or an existing model, such as betting-market odds.